<a href="https://colab.research.google.com/github/yoshiagent/IntControl-colab/blob/main/notebooks/PID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# PID 控制：原課程 MATLAB 程式與 Simulink 模型的 Python 版

對照的網頁：[PID 控制](https://yoshiagent.github.io/IntControl/PID.html)、[Simulink 示範](https://yoshiagent.github.io/IntControl/PID-pidRob.html)、
[數位實作](https://yoshiagent.github.io/IntControl/PID-PID_Digital.html)、[用 SISOTool 設計](https://yoshiagent.github.io/IntControl/PID-SISOTool.html)。

| 原講義附檔 | 內容 | 本筆記本 |
|---|---|---|
| DCMotorData.m、DCMotorDataPID.m | 直流馬達參數、PID 增益 | 第 1 節 |
| DCMotorLoadOpenL.mdl | 馬達開迴路＋負載 | 第 2 節 |
| DCMotorTfSubSPID.mdl | 馬達＋PID（轉速步階響應） | 第 3 節 |
| DCMotorLoadPID.mdl | 馬達＋PID＋負載 | 第 4 節 |
| pid.mdl | 自建的 PID 子系統，以訊號產生器驅動 | 第 5 節 |
| mylib.mdl、MCKPID.mdl、MCKPIDDetail.mdl | MCK 系統的 PID 位置控制 | 第 6 節 |
| （網頁的虛擬碼） | 數位 PID：取樣、梯形積分、輸出限制 | 第 7 節 |
| （網頁的例子） | Ziegler–Nichols 兩種調整法、SISOTool 設計結果 | 第 8、9 節 |

轉移函數用 `scipy.signal`（相當於 MATLAB 的 `tf`、`step`、`lsim`），有負載、飽和等非線性的模型用四階 Runge–Kutta 積分。只用 Colab 內建的 numpy、scipy 與 plotly。

In [1]:
import math
import numpy as np
from scipy import signal
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def step_info(t, y, final=1.0):
    """超越量 %、峰值、峰值時間、2% 安定時間"""
    ip = int(np.argmax(y))
    out = np.where(np.abs(y - final) > 0.02 * abs(final))[0]
    ts = t[out[-1] + 1] if len(out) and out[-1] + 1 < len(t) else (math.nan if len(out) else 0.0)
    return dict(os=max(0.0, (y[ip] - final) / final * 100), peak=y[ip], tp=t[ip], ts=ts)

def ts_text(m):
    return "模擬時間內沒有進入 ±2%" if math.isnan(m["ts"]) else f"{m['ts']:.2f} s"

def xy(x, y, n=600):
    """畫圖用：最多取 n 點、取到小數 5 位（計算本身仍用完整的資料），讓筆記本檔案小一點"""
    x, y = np.asarray(x), np.asarray(y)
    k = max(1, len(x) // n)
    return dict(x=np.round(x[::k], 5), y=np.round(y[::k], 5))

## 1. 直流馬達的參數：DCMotorData.m、DCMotorDataPID.m

```matlab
% DCMotorData.m
L=0.5;
B=0.1;
R=1;
Jm=0.01;
Ke=0.01;
Kt=0.01;
```
```matlab
% DCMotorDataPID.m（再加上三個 PID 增益）
Kp=100;
Ki=200;
Kd=10;
```
馬達的方程式：L di/dt = V − R i − Ke ω，Jm dω/dt = Kt i − B ω − T_L。消去電流得電壓到轉速的轉移函數

$$\frac{\Omega(s)}{V(s)} = \frac{K_t}{(Ls + R)(J_m s + B) + K_e K_t}$$

In [2]:
L, B, R, Jm, Ke, Kt = 0.5, 0.1, 1.0, 0.01, 0.01, 0.01
Kp, Ki, Kd = 100.0, 200.0, 10.0

num_G = [Kt]
den_G = np.polyadd(np.polymul([L, R], [Jm, B]), [Ke * Kt])
G = signal.lti(num_G, den_G)
print("G(s) =", num_G, "/", np.round(den_G, 6))
print("穩態增益 Kt/(RB + KeKt) =", round(Kt / (R * B + Ke * Kt), 4), "(rad/s)/V")

G(s) = [0.01] / [0.005  0.06   0.1001]
穩態增益 Kt/(RB + KeKt) = 0.0999 (rad/s)/V


## 2. 開迴路加負載：DCMotorLoadOpenL.mdl

模型的 DC Motor 子系統直接用狀態方程式（Gain：B/Jm、Kt/Jm、1/L、R/L、Ke/L 加上積分器）建成。輸入電壓是 t = 0 時的 0.6 V 步階（方塊名稱 Speed Setpoint），負載方塊 TL 在 t = 4 s 由 0 跳到 3，
加在角加速度上（相當於 T_L/Jm = 3 rad/s²，即 T_L = 0.03 N·m）。

In [3]:
def motor_sim(controller=None, r=0.6, tl_acc=3.0, t_load=4.0, t_end=10.0, dt=1e-3):
    """直接積分馬達的狀態 (i, ω, ∫e)。controller=None：開迴路，電壓 = r；否則 (Kp, Ki, Kd)，微分作用在量測值"""
    i = w = I = 0.0
    ts = np.arange(0, t_end + dt / 2, dt)
    ws = np.empty_like(ts)
    for k, t in enumerate(ts):
        ws[k] = w
        d = tl_acc if t >= t_load else 0.0
        def f(i_, w_, I_):
            wdot = (Kt * i_ - B * w_) / Jm - d
            u = r if controller is None else controller[0] * (r - w_) + controller[1] * I_ - controller[2] * wdot
            return np.array([(u - R * i_ - Ke * w_) / L, wdot, r - w_])
        x = np.array([i, w, I])
        a = f(*x); b = f(*(x + dt / 2 * a)); c = f(*(x + dt / 2 * b)); e = f(*(x + dt * c))
        i, w, I = x + dt / 6 * (a + 2 * b + 2 * c + e)
    return ts, ws

t, w = motor_sim(None)
print(f"t = 4 s 前轉速 {w[np.searchsorted(t, 3.99)]:.4f} rad/s，加負載後 {w[-1]:.4f} rad/s")
fig = go.Figure(go.Scatter(x=t[::10], y=w[::10], mode="lines", name="omega"))
fig.add_vline(x=4, line_dash="dot", annotation_text="load T_L applied")
fig.update_layout(title="DCMotorLoadOpenL: open loop, 0.6 V, load at t = 4 s", xaxis_title="t (s)", yaxis_title="speed (rad/s)", height=340)
fig.show()

t = 4 s 前轉速 0.0599 rad/s，加負載後 -0.2398 rad/s


輸入 0.6 V，轉速只到 0.6 × 0.0999 ≈ 0.060 rad/s；加上負載後掉了 R·T_L/(RB + KeKt) ≈ 0.30 rad/s，變成 −0.24 rad/s，馬達被負載拖著反轉。

## 3. 加上 PID：DCMotorTfSubSPID.mdl

模型用 Simulink Extras 的 **PID Controller** 方塊（參數 P = Kp、I = Ki、D = Kd，取自 DCMotorDataPID.m），設定值是 t = 0 的單位步階，模擬 3 秒。
Simulink 的步階方塊在 t = 0 時一開始就輸出 1，微分方塊看不到那一下跳動，所以結果相當於「微分只作用在量測值」：

$$U = K_p E + \frac{K_i}{s}E - K_d s\,\Omega \quad\Longrightarrow\quad \frac{\Omega}{R} = \frac{(K_p s + K_i)\,G}{s + (K_d s^2 + K_p s + K_i)\,G}$$

網頁上比較了三組增益：

In [4]:
def closed_loop_dpv(kp, ki, kd):
    """微分作用在量測值的閉迴路"""
    num = np.polymul([kp, ki], num_G)
    den = np.polyadd(np.polymul([1, 0], den_G), np.polymul([kd, kp, ki], num_G))
    return signal.lti(num, den)

def closed_loop_ideal(kp, ki, kd):
    """微分作用在誤差（C = (Kd s² + Kp s + Ki)/s）的閉迴路，作為對照"""
    n = np.polymul([kd, kp, ki], num_G)
    return signal.lti(n, np.polyadd(np.polymul([1, 0], den_G), n))

tt = np.linspace(0, 3, 3001)
fig = go.Figure()
for gains in ((100, 1, 1), (100, 200, 1), (100, 200, 10)):
    _, y = signal.step(closed_loop_dpv(*gains), T=tt)
    m = step_info(tt, y)
    print(f"Kp={gains[0]}, Ki={gains[1]}, Kd={gains[2]}：峰值 {m['peak']:.3f}（{m['tp']:.2f} s），超越 {m['os']:.1f}%，"
          f"2% 安定 {ts_text(m)}，3 s 時 {y[-1]:.3f}")
    fig.add_scatter(**xy(tt[::5], y[::5]), mode="lines", name=f"Kp={gains[0]}, Ki={gains[1]}, Kd={gains[2]}")
_, yi = signal.step(closed_loop_ideal(100, 200, 10), T=tt)
print(f"對照：微分作用在誤差時（100, 200, 10）超越 {step_info(tt, yi)['os']:.1f}%")
fig.add_scatter(**xy(tt[::5], yi[::5]), mode="lines", line=dict(dash="dash"), name="Kd on error (100, 200, 10)")
fig.update_layout(title="DCMotorTfSubSPID: speed step response", xaxis_title="t (s)", yaxis_title="speed (rad/s)", height=380)
fig.show()

Kp=100, Ki=1, Kd=1：峰值 1.079（0.24 s），超越 7.9%，2% 安定 模擬時間內沒有進入 ±2%，3 s 時 0.912
Kp=100, Ki=200, Kd=1：峰值 1.247（0.25 s），超越 24.7%，2% 安定 0.58 s，3 s 時 1.000
Kp=100, Ki=200, Kd=10：峰值 1.116（0.54 s），超越 11.6%，2% 安定 1.35 s，3 s 時 1.000
對照：微分作用在誤差時（100, 200, 10）超越 1.0%


* Ki = 1：積分太弱，3 s 時只到 0.91。
* Ki 加到 200：穩態誤差很快歸零，但超越約 25%。
* 再把 Kd 加到 10：超越降到約 12%，約 1.3 s 安定。
* 若微分真的作用在誤差上（虛線），設定值的步階會在 t = 0⁺ 產生一個電壓脈衝，曲線幾乎沒有超越，但實際的驅動器送不出這種脈衝。

## 4. PID 加負載：DCMotorLoadPID.mdl

自建的 PID 子系統（遮罩參數 P|I|D = 100|200|10），設定 0.6 rad/s、t = 4 s 加上和第 2 節相同的負載：

In [5]:
t, w = motor_sim((100, 200, 10))
t0, w0 = motor_sim(None)
print(f"PID：加負載前 {w[np.searchsorted(t, 3.99)]:.4f}、最後 {w[-1]:.4f} rad/s（最低 {w[t > 4].min():.4f}）")
fig = go.Figure()
fig.add_scatter(**xy(t[::10], w[::10]), mode="lines", name="PID (100, 200, 10)")
fig.add_scatter(**xy(t0[::10], w0[::10]), mode="lines", line=dict(dash="dot"), name="open loop (section 2)")
fig.add_vline(x=4, line_dash="dot")
fig.update_layout(title="DCMotorLoadPID: set point 0.6 rad/s, load at t = 4 s", xaxis_title="t (s)", yaxis_title="speed (rad/s)", height=340)
fig.show()

PID：加負載前 0.6000、最後 0.6000 rad/s（最低 0.5225）


PID 迴路只讓轉速短暫下降，積分項很快就把它拉回 0.6 rad/s。

## 5. 自建 PID 子系統：pid.mdl

pid.mdl 以訊號產生器（正弦波，振幅 1、頻率 0.1 rad/s）驅動自建的 PID 子系統：輸入分別乘上 P、經 I 增益後積分、經微分後乘上 D，三項相加。
遮罩參數 P|I|D = 2|0|0，所以輸出就是 2 倍的輸入。下面也畫出把 I、D 打開時三項各自的輸出（補充）：

In [6]:
t = np.linspace(0, 10, 1001)
e = np.sin(0.1 * t)
def pid_block(e, t, P, I, D):
    integ = np.concatenate([[0], np.cumsum((e[1:] + e[:-1]) / 2 * np.diff(t))])
    deriv = np.gradient(e, t)
    return P * e, I * integ, D * deriv

fig = make_subplots(rows=1, cols=2, subplot_titles=("pid.mdl as shipped: P=2, I=0, D=0", "P=2, I=1, D=5 (each term)"))
p, i_, d = pid_block(e, t, 2, 0, 0)
fig.add_scatter(**xy(t, e), mode="lines", name="input sin(0.1t)", row=1, col=1)
fig.add_scatter(**xy(t, p + i_ + d), mode="lines", name="PID output", row=1, col=1)
p, i_, d = pid_block(e, t, 2, 1, 5)
for y, nm in ((p, "P term"), (i_, "I term"), (d, "D term"), (p + i_ + d, "sum")):
    fig.add_scatter(**xy(t, y), mode="lines", name=nm, row=1, col=2)
fig.update_xaxes(title_text="t (s)"); fig.update_layout(height=360)
fig.show()

比例項跟著誤差走；積分項是誤差的累積（落後）；微分項是誤差的變化率（超前）。

## 6. MCK 系統的 PID 位置控制：mylib.mdl、MCKPID.mdl、MCKPIDDetail.mdl

mylib.mdl 是自建的方塊庫：`MCK_System`（m ẍ + c ẋ + k x = f，遮罩參數 m|c|k = 1|1|2）與 `PID` 子系統。

* **MCKPID.mdl**：步階（預設在 t = 1 s 由 0 跳到 1）→ 誤差 → PID（遮罩參數 P|I|D = 6|2|0，也就是 PI 控制）→ MCK，模擬 20 s。
* **MCKPIDDetail.mdl**：把 PID 展開在最上層，P、I、D 三個 Gain 方塊都保留預設值 1，步階在 t = 0，比例、積分、微分三項各接一個示波器。

設定值固定時微分 de/dt = −ẋ，閉迴路轉移函數為 X/R = (P s + I) / (m s³ + (c + D) s² + (k + P) s + I)。

In [7]:
m_, c_, k_ = 1.0, 1.0, 2.0

def mck_pid(P, I, D, t_end=20.0, t_step=0.0, n=4001):
    """步階在 t_step：t_step 之前 x = 0，之後是平移 t_step 的步階響應（精確，不受取樣格點影響）"""
    t = np.linspace(0, t_end, n)
    r = (t >= t_step).astype(float)
    sys_ = signal.lti([P, I], [m_, c_ + D, k_ + P, I])
    x = np.zeros_like(t)
    k = t >= t_step
    _, x[k] = signal.step(sys_, T=t[k] - t_step)
    return t, r, x

t, r, x = mck_pid(6, 2, 0, t_step=1.0)
mi = step_info(t[t >= 1] - 1, x[t >= 1])
print(f"MCKPID（P=6, I=2, D=0）：峰值 {mi['peak']:.3f}，超越 {mi['os']:.1f}%，步階後 2% 安定 {mi['ts']:.2f} s，20 s 時 x = {x[-1]:.4f}")
fig = go.Figure()
fig.add_scatter(**xy(t, r), mode="lines", line=dict(dash="dash", color="gray"), name="step (t = 1 s)")
fig.add_scatter(**xy(t, x), mode="lines", name="x, PID 6|2|0")
for P, I, D in ((6, 0, 0), (6, 2, 3)):
    fig.add_scatter(**xy(t, mck_pid(P, I, D, t_step=1.0)[2]), mode="lines", name=f"x, PID {P}|{I}|{D}")
fig.update_layout(title="MCKPID: position control of the MCK system", xaxis_title="t (s)", yaxis_title="x", height=360)
fig.show()

MCKPID（P=6, I=2, D=0）：峰值 1.326，超越 32.6%，步階後 2% 安定 11.64 s，20 s 時 x = 0.9986


只有比例（6|0|0）時穩態 x = P/(k + P) = 0.75，留有誤差；加上積分（6|2|0）就會回到 1；再加微分（6|2|3）可以壓低超越量。

MCKPIDDetail.mdl（P = I = D = 1）的三個示波器分別看比例、積分、微分三項：

In [8]:
t, r, x = mck_pid(1, 1, 1)
e = r - x
xdot = np.gradient(x, t)
integ = np.concatenate([[0], np.cumsum((e[1:] + e[:-1]) / 2 * np.diff(t))])
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("x (Scope)", "P, I, D terms (ScopeP, ScopeI, ScopeD)"))
fig.add_scatter(**xy(t, r), mode="lines", line=dict(dash="dash", color="gray"), name="set point", row=1, col=1)
fig.add_scatter(**xy(t, x), mode="lines", name="x", row=1, col=1)
fig.add_scatter(**xy(t, 1 * e), mode="lines", name="P·e", row=2, col=1)
fig.add_scatter(**xy(t, 1 * integ), mode="lines", name="I·∫e", row=2, col=1)
fig.add_scatter(**xy(t, -1 * xdot), mode="lines", name="D·de/dt", row=2, col=1)
fig.update_xaxes(title_text="t (s)", row=2, col=1); fig.update_layout(height=520)
fig.show()
print(f"20 s 時：x = {x[-1]:.4f}，比例項 {e[-1]:.4f}，積分項 {integ[-1]:.4f}（穩態時積分項負責全部的 k·x = 2）")

20 s 時：x = 0.9999，比例項 0.0001，積分項 1.9997（穩態時積分項負責全部的 k·x = 2）


到了穩態，誤差（比例項）和微分項都趨近 0，維持 x = 1 所需的力 k·x = 2 全部由積分項提供，這就是積分能消除穩態誤差的原因。

## 7. 數位 PID（網頁的虛擬碼）

[數位實作](https://yoshiagent.github.io/IntControl/PID-PID_Digital.html)一頁的演算法：

```
e_old = 0; S = 0                     # 初始化
每隔 T 秒：
    e = SP - PV                       # 誤差
    S = S + Ki * (e + e_old) / 2 * T  # 梯形積分
    D = -Kd * (PV - PV_old) / T       # 只對量測值微分
    u = Kp * e + S + D
    if u > u_max: u = u_max; S = S - Ki * (e + e_old) / 2 * T   # 飽和時退回這次積分
    if u < u_min: u = u_min; S = S - Ki * (e + e_old) / 2 * T
    輸出 u（D/A）；e_old = e；PV_old = PV
```

補充：用它控制第 3 節的馬達（取樣 T = 0.01 s，增益 100|200|10），比較 (a) 沒有電壓限制、(b) 電壓限制 ±12 V 並退回積分（防積分飽和）、(c) 限制 ±12 V 但不退回積分：

In [9]:
def digital_pid_motor(T=0.01, u_lim=None, anti_windup=True, t_end=3.0, sp=1.0, dt=1e-4):
    i = w = 0.0
    S, e_old, pv_old = 0.0, 0.0, 0.0
    ts, ws, us = [], [], []
    u = 0.0
    n_sub = int(round(T / dt))
    for k in range(int(round(t_end / T))):
        pv = w
        e = sp - pv
        dS = Ki * (e + e_old) / 2 * T
        S += dS
        D = -Kd * (pv - pv_old) / T if k > 0 else 0.0
        u = Kp * e + S + D
        if u_lim is not None and abs(u) > u_lim:
            u = math.copysign(u_lim, u)
            if anti_windup:
                S -= dS
        e_old, pv_old = e, pv
        for _ in range(n_sub):                                # 零階保持：這 T 秒內電壓固定為 u
            ts.append(len(ts) * dt); ws.append(w); us.append(u)
            f = lambda i_, w_: np.array([(u - R * i_ - Ke * w_) / L, (Kt * i_ - B * w_) / Jm])
            x = np.array([i, w]); a = f(*x); b = f(*(x + dt / 2 * a)); c = f(*(x + dt / 2 * b)); d = f(*(x + dt * c))
            i, w = x + dt / 6 * (a + 2 * b + 2 * c + d)
    return np.array(ts), np.array(ws), np.array(us)

tc, yc = tt, signal.step(closed_loop_dpv(100, 200, 10), T=tt)[1]
cases = {"(a) digital, no limit": dict(), "(b) ±12 V, anti-windup": dict(u_lim=12), "(c) ±12 V, no anti-windup": dict(u_lim=12, anti_windup=False)}
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("speed", "voltage u"))
fig.add_scatter(**xy(tc[::10], yc[::10]), mode="lines", line=dict(dash="dash", color="gray"), name="continuous (section 3)", row=1, col=1)
for name, kw in cases.items():
    t, w, u = digital_pid_motor(**kw)
    m = step_info(t, w)
    print(f"{name}：峰值 {m['peak']:.3f}、超越 {m['os']:.1f}%、2% 安定 {ts_text(m)}、最大電壓 {np.abs(u).max():.1f} V")
    fig.add_scatter(**xy(t[::20], w[::20]), mode="lines", name=name, legendgroup=name, row=1, col=1)
    fig.add_scatter(**xy(t[::20], u[::20]), mode="lines", name=name, legendgroup=name, showlegend=False, row=2, col=1)
fig.update_yaxes(range=[-20, 110], row=2, col=1); fig.update_xaxes(title_text="t (s)", row=2, col=1); fig.update_layout(height=560)
fig.show()

(a) digital, no limit：峰值 1.114、超越 11.4%、2% 安定 1.35 s、最大電壓 101.0 V


(b) ±12 V, anti-windup：峰值 1.000、超越 0.0%、2% 安定 1.24 s、最大電壓 12.0 V


(c) ±12 V, no anti-windup：峰值 1.194、超越 19.4%、2% 安定 模擬時間內沒有進入 ±2%、最大電壓 12.0 V


* (a) 取樣 0.01 s 的數位 PID，峰值與安定時間和連續的結果幾乎相同；上升段有一點差異：電壓每 0.01 s 才更新一次（零階保持），微分項的「煞車」慢了一步，所以一開始反而上升得稍快一點。
* 增益 Kp = 100 時，設定值一變，控制器一開始就要 100 V，實際的驅動器做不到。限制在 ±12 V 後響應變慢。
* (c) 飽和時若還繼續積分，積分項會一直累積（積分飽和，windup），等轉速到了設定值還要很久才「放掉」，造成很大的超越；(b) 退回這次的積分就能避免。

## 8. Ziegler–Nichols 調整法（範例受控體 1/(s+1)³）

### 反應曲線法

對受控體做步階測試，在反曲點（斜率最大處）畫切線，量出延遲時間 L 與時間常數 T，K = 1，K_c = T/(K L)，PID 取 K_p = 1.2 K_c、T_i = 2L、T_d = 0.5L：

In [10]:
P3 = signal.lti([1], [1, 3, 3, 1])
t = np.linspace(0, 15, 15001)
_, y = signal.step(P3, T=t)
dy = np.gradient(y, t)
k = int(np.argmax(dy))                                         # 反曲點
Lz = t[k] - y[k] / dy[k]                                       # 切線和 y = 0 的交點
Tz = t[k] + (1 - y[k]) / dy[k] - Lz                            # 切線到達 1 的時間 − L
Kc = Tz / (1 * Lz)
print(f"反曲點 t = {t[k]:.3f} s，L = {Lz:.3f} s，T = {Tz:.3f} s，Kc = {Kc:.2f}")
zn1 = dict(Kp=1.2 * Kc, Ti=2 * Lz, Td=0.5 * Lz)
print("反應曲線法 PID：Kp = {Kp:.2f}, Ti = {Ti:.2f} s, Td = {Td:.2f} s".format(**zn1))
fig = go.Figure()
fig.add_scatter(**xy(t[::20], y[::20]), mode="lines", name="step response of 1/(s+1)^3")
fig.add_scatter(x=[Lz, Lz + Tz], y=[0, 1], mode="lines", line=dict(dash="dash"), name="tangent at inflection point")
fig.update_layout(title="Ziegler–Nichols reaction curve", xaxis_title="t (s)", yaxis_title="y", height=340)
fig.show()

反曲點 t = 2.000 s，L = 0.805 s，T = 3.695 s，Kc = 4.59
反應曲線法 PID：Kp = 5.50, Ti = 1.61 s, Td = 0.40 s


### 臨界增益法

只用比例控制，逐步加大增益直到持續等幅振盪。閉迴路特徵方程式 s³ + 3s² + 3s + 1 + K = 0，由羅斯判別法得臨界增益 K_u = 8（3·3 = 1 + K），此時極點在 ±j√3，週期 T_u = 2π/√3。
下面用模擬找：

In [11]:
for K in (4, 6, 8, 9):
    print(f"K = {K}：閉迴路極點 {np.round(np.roots([1, 3, 3, 1 + K]), 3)}")
t = np.linspace(0, 40, 40001)
_, y = signal.step(signal.lti([8], [1, 3, 3, 9]), T=t)
pk = [i for i in range(1, len(y) - 1) if y[i] > y[i - 1] and y[i] >= y[i + 1] and t[i] > 5]
Tu = t[pk[2]] - t[pk[1]]
print(f"K = 8 時振盪週期 {Tu:.3f} s（理論 2π/√3 = {2 * math.pi / math.sqrt(3):.3f} s）")
zn2 = dict(Kp=0.6 * 8, Ti=0.5 * Tu, Td=0.125 * Tu)
print("臨界增益法 PID：Kp = {Kp:.2f}, Ti = {Ti:.2f} s, Td = {Td:.2f} s".format(**zn2))

K = 4：閉迴路極點 [-2.587+0.j    -0.206+1.375j -0.206-1.375j]
K = 6：閉迴路極點 [-2.817+0.j    -0.091+1.574j -0.091-1.574j]
K = 8：閉迴路極點 [-3.+0.j    -0.+1.732j -0.-1.732j]
K = 9：閉迴路極點 [-3.08+0.j     0.04+1.801j  0.04-1.801j]
K = 8 時振盪週期 3.628 s（理論 2π/√3 = 3.628 s）
臨界增益法 PID：Kp = 4.80, Ti = 1.81 s, Td = 0.45 s


In [12]:
t = np.linspace(0, 20, 4001)
fig = go.Figure()
for name, z in (("reaction curve", zn1), ("ultimate gain", zn2)):
    nc = np.array([z["Kp"] * z["Td"], z["Kp"], z["Kp"] / z["Ti"]])
    n = nc
    d = np.polyadd(np.polymul([1, 0], [1, 3, 3, 1]), nc)
    _, y = signal.step(signal.lti(n, d), T=t)
    m = step_info(t, y)
    print(f"{name}：超越 {m['os']:.0f}%，2% 安定 {m['ts']:.1f} s")
    fig.add_scatter(**xy(t, y), mode="lines", name=f"{name}: Kp={z['Kp']:.2f}, Ti={z['Ti']:.2f}, Td={z['Td']:.2f}")
fig.update_layout(title="PID tuned by Ziegler–Nichols on 1/(s+1)^3", xaxis_title="t (s)", yaxis_title="y", height=360)
fig.show()

reaction curve：超越 52%，2% 安定 13.0 s
ultimate gain：超越 41%，2% 安定 9.4 s


Ziegler–Nichols 的目標是「每週期衰減到四分之一」，參數偏積極，超越量不小，只能當起點。
（這裡和 PID 頁的圖一樣，用理想式 C(s) = Kp(1 + 1/(Ti s) + Td s)，微分作用在誤差上。[現代最佳化](https://yoshiagent.github.io/IntControl/OPT.html)一章的模擬改成微分作用在量測值、加上 ±10 的輸出限制，同一組臨界增益法參數的超越量是 53%，所以數字不同。）用最佳化自動調整的方法見 [現代最佳化與調參](https://yoshiagent.github.io/IntControl/OPT.html) 一章。

## 9. SISOTool 的設計結果

[用 SISOTool 設計](https://yoshiagent.github.io/IntControl/PID-SISOTool.html)一頁在 MATLAB 的 SISOTool 中為 G(s) = 1/(s² + s + 2) 設計出 C(s) = 5.42 (1 + 0.94 s + (0.55 s)²) / s，
也就是 Kp = 5.42 × 0.94、Ki = 5.42、Kd = 5.42 × 0.55²。SISOTool 是圖形介面，這裡只驗證設計結果：

In [13]:
nc = 5.42 * np.array([0.55 ** 2, 0.94, 1.0])
n = nc
d = np.polyadd(np.polymul([1, 0], [1, 1, 2]), nc)
t = np.linspace(0, 7, 7001)
_, y = signal.step(signal.lti(n, d), T=t)
m = step_info(t, y)
print(f"Kp = {5.42 * 0.94:.2f}, Ki = 5.42, Kd = {5.42 * 0.55 ** 2:.2f}；控制器零點 {np.round(np.roots([0.55 ** 2, 0.94, 1]), 3)}")
print(f"閉迴路極點 {np.round(np.roots(d), 3)}")
print(f"峰值 {m['peak']:.3f}（{m['tp']:.2f} s），超越 {m['os']:.1f}%")
fig = go.Figure(go.Scatter(**xy(t, y), mode="lines"))
fig.update_layout(title="SISOTool design: C = 5.42(1 + 0.94s + (0.55s)^2)/s, G = 1/(s^2+s+2)", xaxis_title="t (s)", yaxis_title="y", height=340)
fig.show()

Kp = 5.09, Ki = 5.42, Kd = 1.64；控制器零點 [-1.554+0.944j -1.554-0.944j]
閉迴路極點 [-0.823+2.187j -0.823-2.187j -0.993+0.j   ]
峰值 1.130（1.13 s），超越 13.0%
